# 01 · Ingestion & Data-Source Analysis

**Goal:** understand *where* the data comes from and *what* it looks like before we touch it.

We use **DuckDB** as the engine (see [`docs/why-duckdb.html`](../docs/why-duckdb.html)). DuckDB's `httpfs` extension
reads remote Parquet **directly** — no AWS account, no download, no staging. The TLC data is public.

### Data sources in play
| Source | What | Access |
|---|---|---|
| CloudFront mirror | `yellow_tripdata_2023-MM.parquet` × 12 | public HTTPS |
| AWS Open Data | `s3://nyc-tlc/trip data/...` | anonymous S3 (`--no-sign-request`) |
| Zone lookup (dim) | `taxi_zone_lookup.csv` (265 rows) | public HTTPS |

> Run `pip install -r ../requirements.txt` once before this notebook.

In [ ]:
import duckdb, pandas as pd

con = duckdb.connect()                 # in-memory DB; nothing persisted yet
con.sql("INSTALL httpfs; LOAD httpfs;")  # the 'connector' that reads remote Parquet
print('DuckDB', duckdb.__version__, '— httpfs loaded')

## 1. Define the sources
We point at the CloudFront mirror by default. The S3 equivalent is shown so you can swap it in.

In [ ]:
BASE   = 'https://d37ci6vzurychx.cloudfront.net/trip-data'
MONTHS = [f'{m:02d}' for m in range(1, 13)]
FILES  = [f'{BASE}/yellow_tripdata_2023-{m}.parquet' for m in MONTHS]
ZONES  = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

# S3 alternative (anonymous):  s3://nyc-tlc/trip data/yellow_tripdata_2023-01.parquet
# In DuckDB: SET s3_region='us-east-1'; then read 's3://nyc-tlc/trip data/yellow_tripdata_2023-*.parquet'

ONE = FILES[0]   # use one month for fast exploration; full year = FILES (glob)
print('One month  :', ONE)
print('Full glob  :', f'{BASE}/yellow_tripdata_2023-*.parquet')

## 2. Inspect the schema — without downloading
`DESCRIBE` reads only the Parquet footer (metadata), so this is near-instant even on a remote file.

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{ONE}')").df()

## 3. Peek at real rows
A `LIMIT` query streams only the first row-group — cheap.

In [ ]:
con.sql(f"""
  SELECT tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count,
         trip_distance, PULocationID, DOLocationID,
         fare_amount, tip_amount, total_amount, payment_type
  FROM read_parquet('{ONE}')
  LIMIT 8
""").df()

## 4. Volume per source file
Row counts use Parquet metadata where possible — fast. This confirms the ~38M-row, 12-file shape.

> Counting all 12 remotely takes a minute or two. Set `FULL_YEAR = True` to do it; otherwise we count one month.

In [ ]:
FULL_YEAR = False
targets = list(zip(MONTHS, FILES)) if FULL_YEAR else [(MONTHS[0], FILES[0])]

rows = []
for m, f in targets:
    n = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{f}')").fetchone()[0]
    rows.append({'month': f'2023-{m}', 'row_count': n})
    print(f'2023-{m}: {n:,} rows')

vol = pd.DataFrame(rows)
vol.loc['TOTAL'] = ['ALL', vol['row_count'].sum()]
vol

## 5. The dimension source — zone lookup
265 rows mapping `LocationID` → Borough / Zone / service_zone. There is **no lat/long** in the trip data,
so this CSV is how trips get geography. We join it in notebook `03`.

In [ ]:
zones = con.sql(f"SELECT * FROM read_csv_auto('{ZONES}')").df()
print(f'{len(zones)} zones across {zones.Borough.nunique()} boroughs')
zones.head(6)

## Takeaways
- **No credentials needed** — public CloudFront/S3 Parquet, read in place via `httpfs`.
- The dataset is **batch** (monthly publish, ~2-month lag) → a monthly job is the right cadence in production.
- Schema matches the brief; geography lives only in the **zone-lookup dimension**.
- Next: **`02_data_quality`** — quantify how many rows the required filters remove, and why.